<a href="https://colab.research.google.com/github/manaskanugo97-max/Healthcare-Lead-Gen-System/blob/main/online_presence_checker_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/healthcare_lead_gen_project

/content/drive/MyDrive/healthcare_lead_gen_project


In [ ]:
import os
print(os.getcwd())

/content/drive/MyDrive/healthcare_lead_gen_project


In [ ]:
print(os.listdir("data"))

['healthcare_osm_raw.csv']


In [ ]:
%%writefile online_presence_checker.py

Writing online_presence_checker.py


In [ ]:
from config import RAW_OSM_CSV, ONLINE_PRESENCE_CSV

In [ ]:
import pandas as pd
df = pd.read_csv(RAW_OSM_CSV)

In [ ]:
df.to_csv(ONLINE_PRESENCE_CSV, index=False)

In [ ]:
RAW_OSM_CSV
ONLINE_PRESENCE_CSV

'/content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_online_presence.csv'

In [ ]:
!pip install ddgs duckduckgo-search pandas tldextract -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 72.9 MB/s eta 0:00:00


In [ ]:
%%writefile online_presence_checker.py

import os
import time
import pandas as pd
import tldextract

from config import RAW_OSM_CSV, ONLINE_PRESENCE_CSV

try:
    from ddgs import DDGS
except ImportError:
    from duckduckgo_search import DDGS


REFERRAL_PLATFORMS = [
    "practo",
    "justdial",
    "sulekha",
    "lybrate",
    "sehat",
    "medindia",
    "credihealth",
    "clinicspots",
    "asklaila",
    "indiamart",
    "facebook",
    "instagram",
    "linkedin",
    "youtube",
    "twitter",
    "x.com",
    "google",
    "bing",
    "openstreetmap",
    "mapcarta",
    "wikipedia",
    "wikidata"
]


def clean_value(value):
    """
    Converts empty/NaN values into clean blank text.
    """
    if pd.isna(value):
        return ""
    return str(value).strip()


def get_domain(url):
    """
    Extract domain from website URL.
    Example:
    https://www.apollohospitals.com/contact
    becomes:
    apollohospitals.com
    """
    url = clean_value(url)

    if not url:
        return ""

    try:
        extracted = tldextract.extract(url)

        if extracted.domain and extracted.suffix:
            return f"{extracted.domain}.{extracted.suffix}".lower()

    except Exception:
        return ""

    return ""


def is_referral_platform(url):
    """
    Checks if website is a listing/referral/social platform.
    """
    domain = get_domain(url)

    for platform in REFERRAL_PLATFORMS:
        if platform in domain:
            return True

    return False


def search_online(query, max_results=8):
    """
    Search online using DuckDuckGo.
    """
    results = []

    try:
        ddgs = DDGS()
        search_results = ddgs.text(query, max_results=max_results)

        for item in search_results:
            title = item.get("title", "")
            url = item.get("href", "")
            body = item.get("body", "")

            if url:
                results.append({
                    "title": title,
                    "url": url,
                    "body": body
                })

    except Exception as e:
        print("Search failed")
        print("Query:", query)
        print("Reason:", e)

    return results


def find_online_presence(business_name, location, existing_website):
    """
    Finds official website or referral platform links.
    """

    business_name = clean_value(business_name)
    location = clean_value(location)
    existing_website = clean_value(existing_website)

    if existing_website:
        if is_referral_platform(existing_website):
            return "", "Referral Platform", existing_website, existing_website
        else:
            return existing_website, "Official Website", "", existing_website

    query = f'"{business_name}" "{location}" official website clinic hospital healthcare phone'

    search_results = search_online(query, max_results=8)

    official_website = ""
    referral_links = []
    all_links = []

    for result in search_results:
        url = result["url"]
        all_links.append(url)

        if is_referral_platform(url):
            referral_links.append(url)
        else:
            if not official_website:
                official_website = url

    if official_website:
        return (
            official_website,
            "Official Website Found",
            " | ".join(referral_links[:3]),
            " | ".join(all_links[:5])
        )

    if referral_links:
        return (
            "",
            "Referral Platform Only",
            " | ".join(referral_links[:3]),
            " | ".join(all_links[:5])
        )

    return (
        "",
        "No Strong Online Presence Found",
        "",
        " | ".join(all_links[:5])
    )


def check_online_presence(input_file=RAW_OSM_CSV, output_file=ONLINE_PRESENCE_CSV, limit=20):
    """
    Reads Step 1 CSV and creates Step 2 CSV.
    """

    print("Step 2 started")
    print("Input file:", input_file)
    print("Output file:", output_file)

    if not os.path.exists(input_file):
        raise FileNotFoundError(
            f"Input CSV not found: {input_file}. Please run Step 1 first."
        )

    df = pd.read_csv(input_file)

    if limit is not None:
        df = df.head(limit)

    official_websites = []
    online_presence_types = []
    referral_platform_links = []
    search_result_links = []

    print("Total records to check:", len(df))

    for index, row in df.iterrows():
        business_name = row.get("Business Name", "")
        location = row.get("Location", "")
        existing_website = row.get("Website URL", "")

        print(f"[{index + 1}/{len(df)}] Searching: {business_name}")

        official_website, presence_type, referral_links, all_links = find_online_presence(
            business_name,
            location,
            existing_website
        )

        official_websites.append(official_website)
        online_presence_types.append(presence_type)
        referral_platform_links.append(referral_links)
        search_result_links.append(all_links)

        time.sleep(2)

    df["Official Website Found"] = official_websites
    df["Online Presence Type"] = online_presence_types
    df["Referral Platform Links"] = referral_platform_links
    df["Search Result Links"] = search_result_links

    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("Step 2 completed")
    print("Saved file:", output_file)

    return df


if __name__ == "__main__":
    df = check_online_presence(
        input_file=RAW_OSM_CSV,
        output_file=ONLINE_PRESENCE_CSV,
        limit=100
    )

    print(df.head())

Overwriting online_presence_checker.py


In [ ]:
!python online_presence_checker.py

Step 2 started
Input file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_osm_raw.csv
Output file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_online_presence.csv
Total records to check: 100
[1/100] Searching: Greater Kailash Hospital
[2/100] Searching: Qurban Husain
[3/100] Searching: Shri Aurobindo dental clinic and institute of medical science
[4/100] Searching: Dr. Mutha
[5/100] Searching: Arshid shah
[6/100] Searching: Shubham Hospital
[7/100] Searching: Aditya Nursing Home
[8/100] Searching: Curewell hospital janjeerwala square
[9/100] Searching: kjd
[10/100] Searching: Family Dental Clinic
[11/100] Searching: Yoga Amrutam
[12/100] Searching: Suhan Pathology
[13/100] Searching: Dr. Lal Path Labs
[14/100] Searching: Dr. Vrinda Vashistha
Search failed
Query: "Dr. Vrinda Vashistha" "2-BB, Slice No.-5, Scheme No.-78, Vijay Nagar, 25, E.B, Lower Basement,, Indore" official website clinic hospital healthcare phone
Reason: No results found.
[15

In [ ]:
from config import ONLINE_PRESENCE_CSV
import os

print("Step 2 file exists:", os.path.exists(ONLINE_PRESENCE_CSV))
print("Step 2 file path:", ONLINE_PRESENCE_CSV)

Step 2 file exists: True
Step 2 file path: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_online_presence.csv


In [ ]:
import pandas as pd
from config import ONLINE_PRESENCE_CSV

df2 = pd.read_csv(ONLINE_PRESENCE_CSV)
df2.head()

,Business Name,Industry Category,Business Description,Location,Latitude,Longitude,Google Maps Profile Link,OpenStreetMap Link,Website URL,Phone Number,Email Address,Source,Official Website Found,Online Presence Type,Referral Platform Links,Search Result Links
0,Greater Kailash Hospital,hospital,Hospital listed on OpenStreetMap,Indore,22.725082,75.890465,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/1802056893,NaN,NaN,NaN,OpenStreetMap,https://www.gkh.co.in/,Official Website Found,https://www.medindia.net/directories/hospitals...,https://www.gkh.co.in/ | https://www.hexahealt...
1,Qurban Husain,doctors,Doctors listed on OpenStreetMap,Indore,22.716741,75.863790,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/3360065915,NaN,NaN,NaN,OpenStreetMap,https://indushospital.org.pk/,Official Website Found,https://en.wikipedia.org/wiki/Indus_Hospital_a...,https://en.wikipedia.org/wiki/Indus_Hospital_a...
2,Shri Aurobindo dental clinic and institute of ...,clinic,Clinic listed on OpenStreetMap,Ujjain Road,22.791523,75.845636,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/4273145396,NaN,NaN,NaN,OpenStreetMap,https://hospital.sriaurobindouniversity.edu.in/,Official Website Found,https://www.medindia.net/directories/hospitals...,https://hospital.sriaurobindouniversity.edu.in...
3,Dr. Mutha,doctors,Doctors listed on OpenStreetMap,Gita Bhavan intersection,22.718593,75.885588,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/4310485189,NaN,+9198-26-150111,NaN,OpenStreetMap,https://gitabhawanhospital.org/,Official Website Found,https://www.justdial.com/Indore/Dr-Mutha-Sudar...,https://gitabhawanhospital.org/ | https://www....
4,Arshid shah,pharmacy,Pharmacy listed on OpenStreetMap,"Dhar road, 452001",22.709624,75.827079,https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/4522507595,NaN,8103012446,shah.arshid05@rediffmail.com,OpenStreetMap,https://www.latlong.net/poi/arshid-shah-411182,Official Website Found,NaN,https://www.latlong.net/poi/arshid-shah-411182...
